
# Direct CI benchmark for modified mixed-data KCI vs standard KCI + one-hot


Run it **twice**:

1. In the **modified `cdt15/causal-learn` environment** with `RUN_MODE = "modified"`
2. In the **upstream `py-why/causal-learn` environment** with `RUN_MODE = "onehot"`

The notebook will:

- generate two representative mixed-data settings
- evaluate **Type I error** on known null CI relations
- evaluate **power** on known dependent relations
- compare two bandwidth-selection rules (`empirical`, `median`)
- save raw CSV files and summary tables
- automatically produce a clean comparison table once both method CSV files exist


In [13]:
from pathlib import Path
import inspect
import warnings
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import display

warnings.filterwarnings('ignore')
np.set_printoptions(precision=3, suppress=True)

# ===== User settings =====
RUN_MODE = "modified"   # First run: "modified" in the modified environment; second run: "onehot" in upstream py-why/causal-learn.
ALPHA = 0.05

# Include n=2000, close to the empirical sample size n=2056.
SAMPLE_SIZES = [300, 1000, 2000]

# Include weak/moderate/original-strength alternatives.
# Use [1.0] for a smoke test; use [0.25, 0.5, 1.0] for final results.
EFFECT_STRENGTHS = [0.25, 0.5, 1.0]

N_REPS = 50
WIDTH_SETTINGS = ["empirical", "median"]
SCENARIOS_TO_RUN = ["linear_group1_mixed", "nonlinear_groups12_mixed"]
BASE_SEED = 20260410
OUT_DIR = Path("results_ci_benchmark")
OUT_DIR.mkdir(exist_ok=True)

RAW_OUT = OUT_DIR / f"ci_benchmark_{RUN_MODE}.csv"
SUMMARY_OUT = OUT_DIR / f"ci_benchmark_summary_{RUN_MODE}.csv"
QUERY_OUT = OUT_DIR / f"ci_benchmark_query_summary_{RUN_MODE}.csv"
FAIL_OUT = OUT_DIR / f"ci_benchmark_failures_{RUN_MODE}.csv"

print("Python:", platform.python_version())
print("RUN_MODE:", RUN_MODE)
print("sample sizes:", SAMPLE_SIZES)
print("effect strengths:", EFFECT_STRENGTHS)
print("repetitions:", N_REPS)
print("width settings:", WIDTH_SETTINGS)
print("output directory:", OUT_DIR.resolve())


Python: 3.12.3
RUN_MODE: modified
sample sizes: [300, 1000, 2000]
effect strengths: [0.25, 0.5, 1.0]
repetitions: 50
width settings: ['empirical', 'median']
output directory: /home/jiang/results_ci_benchmark


In [14]:

# Package check
import causallearn
import causallearn.utils.cit as cit_module
from causallearn.utils.cit import CIT

print("causallearn package:", causallearn.__file__)
try:
    has_mixed_kci = "is_discrete" in inspect.getsource(cit_module.KCI)
except Exception:
    has_mixed_kci = None

print("Detected mixed-data KCI support in this environment:", has_mixed_kci)
if RUN_MODE == "modified" and has_mixed_kci is False:
    print("WARNING: RUN_MODE is 'modified' but this environment does not seem to include the mixed-data KCI changes.")
if RUN_MODE == "onehot" and has_mixed_kci is True:
    print("NOTE: this environment appears to include the mixed-data KCI changes. For a strict baseline, use upstream py-why/causal-learn.")


causallearn package: /home/jiang/venvs/kci_mixed/lib/python3.12/site-packages/causallearn/__init__.py
Detected mixed-data KCI support in this environment: True


## Scenario definitions

We use two representative observed-variable settings:

- `linear_group1_mixed`: linear SEM, mixed variables only in group 1
- `nonlinear_groups12_mixed`: nonlinear SEM, mixed variables in groups 1 and 2

This notebook is intentionally **CI-test-level**, so it uses observed-variable SEMs with known conditional independences. The FCI graph-level notebook remains the **graph-level** benchmark.

To address the sample-size and weak-link concern, the SEMs include an `effect_strength` multiplier. 


In [15]:
sigmoid = lambda z: 1.0 / (1.0 + np.exp(-z))


def discretize_tertile(values):
    q1, q2 = np.quantile(values, [1.0 / 3.0, 2.0 / 3.0])
    return np.digitize(values, bins=[q1, q2])


def generate_linear_group1_mixed(n, seed, effect_strength=1.0):
    """Linear observed-variable SEM.

    effect_strength multiplies the directed structural coefficients. Values below
    1.0 create weaker alternatives while preserving the null d-separations.
    """
    rng = np.random.default_rng(seed)
    s = float(effect_strength)

    X1 = rng.normal(0.0, 1.0, n)
    B1 = rng.binomial(1, 0.5, size=n)

    T1 = discretize_tertile(rng.normal(0.0, 1.0, n))

    X2_1 = s * (0.9 * X1 - 0.8 * B1) + rng.normal(0.0, 1.0, n)
    X2_2 = s * (0.4 * B1 - 0.3 * T1) + rng.normal(0.0, 1.0, n)
    X2_3 = s * (0.5 * T1) + rng.normal(0.0, 1.0, n)
    X2_4 = s * (0.3 * X1 + 0.6 * X2_1 + 0.7 * X2_3) + rng.normal(0.0, 1.0, n)

    B3 = rng.binomial(1, sigmoid(s * (0.9 * X2_3 + 1.1 * X2_4)), size=n)

    df = pd.DataFrame(
        {
            "X1": X1,
            "B1": B1,
            "T1": T1,
            "X2_1": X2_1,
            "X2_2": X2_2,
            "X2_3": X2_3,
            "X2_4": X2_4,
            "B3": B3,
        }
    )
    return df


def generate_nonlinear_groups12_mixed(n, seed, effect_strength=1.0):
    """Nonlinear observed-variable SEM with mixed variables in groups 1 and 2."""
    rng = np.random.default_rng(seed)
    s = float(effect_strength)

    e1 = rng.normal(0.0, 1.0, n)
    e2 = rng.normal(0.0, 1.0, n)

    X1 = 0.9 * np.sin(e1) + rng.normal(0.0, 1.0, n)
    B1 = rng.binomial(1, sigmoid(0.8 * (e2 ** 2)), size=n)

    T1 = discretize_tertile(rng.normal(0.0, 1.0, n))

    X2_1 = s * (0.9 * np.sin(B1) - 0.4 * (T1 ** 2)) + rng.normal(0.0, 1.0, n)
    X2_2 = s * (-0.7 * np.cos(T1)) + rng.normal(0.0, 1.0, n)

    B2 = rng.binomial(1, sigmoid(s * (0.8 * (X1 ** 2) - 0.7 * np.cos(1.3 * B1))), size=n)

    Zt2 = s * (
        0.8 * np.sin(X1)
        + 0.6 * np.log1p(np.abs(X2_2))
        - 0.4 * np.tanh(0.7 * B2)
    ) + rng.normal(0.0, 1.0, n)
    T2 = discretize_tertile(Zt2)

    B3 = rng.binomial(1, sigmoid(s * (0.7 * np.sin(X2_2) + 0.5 * (T2 ** 2))), size=n)

    df = pd.DataFrame(
        {
            "X1": X1,
            "B1": B1,
            "T1": T1,
            "B2": B2,
            "X2_1": X2_1,
            "X2_2": X2_2,
            "T2": T2,
            "B3": B3,
        }
    )
    return df


def make_query(name, truth, x, y, z):
    return {"query_name": name, "truth": truth, "x": x, "y": y, "z": list(z)}


SCENARIO_LABELS = {
    "linear_group1_mixed": "Linear: mixed variables only in group 1",
    "nonlinear_groups12_mixed": "Nonlinear: mixed variables in groups 1 and 2",
}

SCENARIOS = {
    "linear_group1_mixed": {
        "generator": generate_linear_group1_mixed,
        "node_names": ["X1", "B1", "T1", "X2_1", "X2_2", "X2_3", "X2_4", "B3"],
        "is_discrete": np.array([False, True, True, False, False, False, False, True], dtype=bool),
        "categorical_levels": {"B1": [0, 1], "T1": [0, 1, 2], "B3": [0, 1]},
        "observed_edges": [
            ("X1", "X2_1"), ("X1", "X2_4"),
            ("B1", "X2_1"), ("B1", "X2_2"),
            ("T1", "X2_2"), ("T1", "X2_3"),
            ("X2_1", "X2_4"), ("X2_3", "X2_4"),
            ("X2_3", "B3"), ("X2_4", "B3"),
        ],
        "queries": [
            make_query("N1_X1_ind_B1", "null", "X1", "B1", []),
            make_query("N2_X1_ind_T1", "null", "X1", "T1", []),
            make_query("N3_B1_ind_X2_3", "null", "B1", "X2_3", []),
            make_query("N4_X2_2_ind_X2_3_given_T1", "null", "X2_2", "X2_3", ["T1"]),
            make_query("N5_X1_ind_B3_given_X2_3_X2_4", "null", "X1", "B3", ["X2_3", "X2_4"]),
            make_query("N6_B1_ind_X2_4_given_X1_X2_1", "null", "B1", "X2_4", ["X1", "X2_1"]),
            make_query("A1_X1_dep_X2_1", "alternative", "X1", "X2_1", []),
            make_query("A2_B1_dep_X2_2", "alternative", "B1", "X2_2", []),
            make_query("A3_T1_dep_X2_3", "alternative", "T1", "X2_3", []),
            make_query("A4_X2_4_dep_B3", "alternative", "X2_4", "B3", []),
            make_query("A5_X2_4_dep_B3_given_X2_3", "alternative", "X2_4", "B3", ["X2_3"]),
            make_query("A6_B1_dep_X2_4_given_X1_X2_3", "alternative", "B1", "X2_4", ["X1", "X2_3"]),
        ],
    },
    "nonlinear_groups12_mixed": {
        "generator": generate_nonlinear_groups12_mixed,
        "node_names": ["X1", "B1", "T1", "B2", "X2_1", "X2_2", "T2", "B3"],
        "is_discrete": np.array([False, True, True, True, False, False, True, True], dtype=bool),
        "categorical_levels": {"B1": [0, 1], "T1": [0, 1, 2], "B2": [0, 1], "T2": [0, 1, 2], "B3": [0, 1]},
        "observed_edges": [
            ("X1", "B2"), ("X1", "T2"),
            ("B1", "B2"), ("B1", "X2_1"),
            ("T1", "X2_1"), ("T1", "X2_2"),
            ("B2", "T2"), ("X2_2", "T2"),
            ("X2_2", "B3"), ("T2", "B3"),
        ],
        "queries": [
            make_query("N1_X1_ind_B1", "null", "X1", "B1", []),
            make_query("N2_X1_ind_T1", "null", "X1", "T1", []),
            make_query("N3_B1_ind_X2_2", "null", "B1", "X2_2", []),
            make_query("N4_T1_ind_B2", "null", "T1", "B2", []),
            make_query("N5_X1_ind_B3_given_X2_2_T2", "null", "X1", "B3", ["X2_2", "T2"]),
            make_query("N6_B1_ind_T2_given_X1_B2", "null", "B1", "T2", ["X1", "B2"]),
            make_query("A1_X1_dep_B2", "alternative", "X1", "B2", []),
            make_query("A2_B1_dep_X2_1", "alternative", "B1", "X2_1", []),
            make_query("A3_T1_dep_X2_2", "alternative", "T1", "X2_2", []),
            make_query("A4_X2_2_dep_B3", "alternative", "X2_2", "B3", []),
            make_query("A5_T2_dep_B3", "alternative", "T2", "B3", []),
            make_query("A6_X1_dep_B3_given_X2_2", "alternative", "X1", "B3", ["X2_2"]),
        ],
    },
}

for scenario_name, spec in SCENARIOS.items():
    print(f"{scenario_name}: {len(spec['queries'])} queries, variables = {spec['node_names']}")


linear_group1_mixed: 12 queries, variables = ['X1', 'B1', 'T1', 'X2_1', 'X2_2', 'X2_3', 'X2_4', 'B3']
nonlinear_groups12_mixed: 12 queries, variables = ['X1', 'B1', 'T1', 'B2', 'X2_1', 'X2_2', 'T2', 'B3']


In [16]:

# Verify that the query labels match d-separation in the observed DAG.
def verify_queries_against_dag(scenarios):
    bad = []
    for scenario_name, spec in scenarios.items():
        G = nx.DiGraph()
        G.add_nodes_from(spec["node_names"])
        G.add_edges_from(spec["observed_edges"])
        for q in spec["queries"]:
            is_null = nx.is_d_separator(G, {q["x"]}, {q["y"]}, set(q["z"]))
            expected_null = (q["truth"] == "null")
            if is_null != expected_null:
                bad.append((scenario_name, q["query_name"], q["truth"], q["x"], q["y"], q["z"]))
    return bad

bad_queries = verify_queries_against_dag(SCENARIOS)
if len(bad_queries) == 0:
    print("All queries were verified against the observed DAGs.")
else:
    print("The following queries do not match the DAG and should be fixed:")
    display(pd.DataFrame(bad_queries, columns=["scenario", "query_name", "truth", "x", "y", "z"]))


All queries were verified against the observed DAGs.


In [18]:
query_design_rows = []

for scenario_name, spec in SCENARIOS.items():
    for q in spec["queries"]:
        query_design_rows.append({
            "scenario": SCENARIO_LABELS[scenario_name],
            "query": q["query_name"],
            "relation": "Null CI" if q["truth"] == "null" else "Dependent",
            "X": q["x"],
            "Y": q["y"],
            "conditioning_set": ", ".join(q["z"]) if len(q["z"]) > 0 else r"\(\varnothing\)",
        })

query_design_df = pd.DataFrame(query_design_rows)
display(query_design_df)

,scenario,query,relation,X,Y,conditioning_set
0,Linear: mixed variables only in group 1,N1_X1_ind_B1,Null CI,X1,B1,\(\varnothing\)
1,Linear: mixed variables only in group 1,N2_X1_ind_T1,Null CI,X1,T1,\(\varnothing\)
2,Linear: mixed variables only in group 1,N3_B1_ind_X2_3,Null CI,B1,X2_3,\(\varnothing\)
3,Linear: mixed variables only in group 1,N4_X2_2_ind_X2_3_given_T1,Null CI,X2_2,X2_3,T1
4,Linear: mixed variables only in group 1,N5_X1_ind_B3_given_X2_3_X2_4,Null CI,X1,B3,"X2_3, X2_4"
5,Linear: mixed variables only in group 1,N6_B1_ind_X2_4_given_X1_X2_1,Null CI,B1,X2_4,"X1, X2_1"
6,Linear: mixed variables only in group 1,A1_X1_dep_X2_1,Dependent,X1,X2_1,\(\varnothing\)
7,Linear: mixed variables only in group 1,A2_B1_dep_X2_2,Dependent,B1,X2_2,\(\varnothing\)
8,Linear: mixed variables only in group 1,A3_T1_dep_X2_3,Dependent,T1,X2_3,\(\varnothing\)
9,Linear: mixed variables only in group 1,A4_X2_4_dep_B3,Dependent,X2_4,B3,\(\varnothing\)



## Representation builders

- `modified` mode: use the raw mixed data and pass `is_discrete` to the revised KCI
- `onehot` mode: one-hot encode categorical variables and use standard upstream KCI


In [5]:

def flatten(groups):
    out = []
    for g in groups:
        out.extend(int(i) for i in g)
    return out


def build_raw_representation(df, spec):
    node_names = spec["node_names"]
    data = df[node_names].to_numpy(dtype=float)
    groups = {name: [i] for i, name in enumerate(node_names)}
    is_discrete = spec["is_discrete"].copy()
    return data, groups, is_discrete


def onehot_encode_with_fixed_levels(df, categorical_levels):
    parts = []
    expanded_to_original = {}
    for col in df.columns:
        if col in categorical_levels:
            cat = pd.Categorical(df[col], categories=categorical_levels[col])
            dummies = pd.get_dummies(cat, prefix=col, prefix_sep='__', dtype=float)
            parts.append(dummies)
            for dummy_col in dummies.columns:
                expanded_to_original[dummy_col] = col
        else:
            parts.append(df[[col]].astype(float))
            expanded_to_original[col] = col
    encoded = pd.concat(parts, axis=1)
    return encoded, expanded_to_original


def build_onehot_representation(df, spec):
    encoded, expanded_to_original = onehot_encode_with_fixed_levels(df[spec["node_names"]], spec["categorical_levels"])
    groups = {
        original: [i for i, col in enumerate(encoded.columns) if expanded_to_original[col] == original]
        for original in spec["node_names"]
    }
    return encoded.to_numpy(dtype=float), groups, None


def get_representation(df, spec, run_mode):
    if run_mode == "modified":
        return build_raw_representation(df, spec)
    if run_mode == "onehot":
        return build_onehot_representation(df, spec)
    raise ValueError(f"Unknown run_mode: {run_mode}")



## CI-test runner



In [6]:
def effect_strength_label(effect_strength):
    if effect_strength <= 0.3:
        return "weak"
    if effect_strength <= 0.6:
        return "moderate-weak"
    if effect_strength <= 1.0:
        return "original-strength"
    return "strong"


def run_queries_for_dataset(df, scenario_name, spec, run_mode, est_width, alpha=0.05):
    data, groups, is_discrete = get_representation(df, spec, run_mode)
    kwargs = {"est_width": est_width}
    if run_mode == "modified":
        kwargs["is_discrete"] = is_discrete

    cit = CIT(data, "kci", **kwargs)
    rows = []
    for q in spec["queries"]:
        x_idx = groups[q["x"]]
        y_idx = groups[q["y"]]
        z_idx = flatten(groups[z] for z in q["z"])
        try:
            p_value = float(cit(x_idx, y_idx, z_idx))
            reject = int(p_value < alpha)
            error_msg = ""
        except Exception as e:
            p_value = np.nan
            reject = np.nan
            error_msg = repr(e)

        rows.append(
            {
                "scenario": scenario_name,
                "query_name": q["query_name"],
                "truth": q["truth"],
                "x": q["x"],
                "y": q["y"],
                "z": str(q["z"]),
                "p_value": p_value,
                "reject": reject,
                "error": error_msg,
                "n_features_x": len(x_idx),
                "n_features_y": len(y_idx),
                "n_features_z": len(z_idx),
            }
        )
    return rows


def run_benchmark(run_mode, scenarios_to_run, sample_sizes, effect_strengths, n_reps, width_settings, alpha, base_seed):
    rng = np.random.default_rng(base_seed)
    all_rows = []

    for scenario_name in scenarios_to_run:
        spec = SCENARIOS[scenario_name]
        print(f"\n=== Scenario: {scenario_name} ===")
        for effect_strength in effect_strengths:
            effect_label = effect_strength_label(float(effect_strength))
            print(f"--- effect_strength = {effect_strength} ({effect_label}) ---")
            for n in sample_sizes:
                dataset_seeds = [int(rng.integers(0, 2**32 - 1)) for _ in range(n_reps)]
                for rep, dataset_seed in enumerate(dataset_seeds, start=1):
                    df = spec["generator"](n=n, seed=dataset_seed, effect_strength=effect_strength)
                    for est_width in width_settings:
                        rows = run_queries_for_dataset(df, scenario_name, spec, run_mode, est_width, alpha)
                        for row in rows:
                            row.update(
                                {
                                    "method": run_mode,
                                    "n": n,
                                    "effect_strength": float(effect_strength),
                                    "effect_label": effect_label,
                                    "rep": rep,
                                    "dataset_seed": dataset_seed,
                                    "alpha": alpha,
                                    "est_width": est_width,
                                    "resampling": "independent_monte_carlo",
                                }
                            )
                        all_rows.extend(rows)
                    if rep == 1 or rep % 5 == 0 or rep == n_reps:
                        print(f"  effect={effect_strength}, n={n}, rep={rep}/{n_reps}")

    return pd.DataFrame(all_rows)


In [ ]:
%%time
raw_results = run_benchmark(
    run_mode=RUN_MODE,
    scenarios_to_run=SCENARIOS_TO_RUN,
    sample_sizes=SAMPLE_SIZES,
    effect_strengths=EFFECT_STRENGTHS,
    n_reps=N_REPS,
    width_settings=WIDTH_SETTINGS,
    alpha=ALPHA,
    base_seed=BASE_SEED,
)

raw_results.to_csv(RAW_OUT, index=False)
print(f"Saved raw results to: {RAW_OUT}")
raw_results.head()



=== Scenario: linear_group1_mixed ===
--- effect_strength = 0.25 (weak) ---
  effect=0.25, n=300, rep=1/50
  effect=0.25, n=300, rep=5/50
  effect=0.25, n=300, rep=10/50
  effect=0.25, n=300, rep=15/50
  effect=0.25, n=300, rep=20/50
  effect=0.25, n=300, rep=25/50
  effect=0.25, n=300, rep=30/50
  effect=0.25, n=300, rep=35/50
  effect=0.25, n=300, rep=40/50
  effect=0.25, n=300, rep=45/50
  effect=0.25, n=300, rep=50/50
  effect=0.25, n=1000, rep=1/50



## Summaries for the current run


In [ ]:
def _coerce_reject_to_numeric(s):
    """Convert reject column to 0/1 after CSV loading.

    Depending on pandas version and whether failed rows are present, the
    reject column may be bool, numeric, or strings such as 'True'/'False'.
    """
    if pd.api.types.is_bool_dtype(s):
        return s.astype(float)

    mapped = s.replace({
        True: 1.0,
        False: 0.0,
        "True": 1.0,
        "False": 0.0,
        "true": 1.0,
        "false": 0.0,
        "TRUE": 1.0,
        "FALSE": 0.0,
        "1": 1.0,
        "0": 0.0,
        1: 1.0,
        0: 0.0,
        "": np.nan,
        "nan": np.nan,
        "NaN": np.nan,
        "None": np.nan,
    })
    return pd.to_numeric(mapped, errors="coerce")


def summarize_results(df):
    df = df.copy()

    if "error" not in df.columns:
        df["error"] = ""

    valid = df[df["error"].fillna("").astype(str) == ""].copy()

    # Coerce important columns after pd.read_csv(..., keep_default_na=False).
    valid["reject_num"] = _coerce_reject_to_numeric(valid["reject"])
    valid = valid.dropna(subset=["reject_num"])

    for col in ["n", "rep", "alpha", "effect_strength"]:
        if col in valid.columns:
            valid[col] = pd.to_numeric(valid[col], errors="coerce")

    if "effect_strength" not in valid.columns:
        valid["effect_strength"] = 1.0
    if "effect_label" not in valid.columns:
        valid["effect_label"] = "original-strength"

    # Keep truth labels as strings so that 'null' is not interpreted as missing.
    if "truth" in valid.columns:
        valid["truth"] = valid["truth"].astype(str)

    group_cols = ["scenario", "n", "effect_strength", "effect_label", "method", "est_width"]

    overall = (
        valid.groupby(group_cols + ["truth"], as_index=False)["reject_num"]
        .mean()
        .rename(columns={"reject_num": "rejection_rate"})
    )

    summary = (
        overall.pivot_table(index=group_cols, columns="truth", values="rejection_rate")
        .reset_index()
        .rename(columns={"null": "type1_error", "alternative": "power"})
        .sort_values(["scenario", "effect_strength", "n", "method", "est_width"])
    )

    query_summary = (
        valid.groupby(group_cols + ["truth", "query_name"], as_index=False)["reject_num"]
        .mean()
        .rename(columns={"reject_num": "rejection_rate"})
        .sort_values(["scenario", "effect_strength", "n", "method", "est_width", "truth", "query_name"])
    )

    failures = df[df["error"].fillna("").astype(str) != ""].copy()
    return summary, query_summary, failures


if "raw_results" in globals():
    summary_current, query_summary_current, failures_current = summarize_results(raw_results)
    summary_current.to_csv(SUMMARY_OUT, index=False)
    query_summary_current.to_csv(QUERY_OUT, index=False)
    failures_current.to_csv(FAIL_OUT, index=False)

    print(f"Saved summary to: {SUMMARY_OUT}")
    print(f"Saved query summary to: {QUERY_OUT}")
    print(f"Saved failures to: {FAIL_OUT}")
    if len(failures_current) > 0:
        print(f"Number of failed query evaluations: {len(failures_current)}")
    else:
        print("No failed query evaluations.")

    summary_display = summary_current.copy()
    summary_display["scenario"] = summary_display["scenario"].map(SCENARIO_LABELS)
    display(summary_display.round(3))
else:
    print("raw_results is not defined. If you only want to compare existing CSV files, continue to the comparison cells below.")


In [ ]:

# Query-level results for the current run
query_display = query_summary_current.copy()
query_display["scenario"] = query_display["scenario"].map(SCENARIO_LABELS)
display(query_display)



## Compare both methods when both CSV files exist



In [ ]:
def load_all_available_results(out_dir):
    paths = [
        out_dir / "ci_benchmark_modified.csv",
        out_dir / "ci_benchmark_onehot.csv",
    ]
    frames = []
    for path in paths:
        if path.exists():
            # Important: do not let pandas treat the string "null" as NaN.
            df = pd.read_csv(path, keep_default_na=False)

            if "truth" in df.columns:
                df["truth"] = df["truth"].astype(str)
            if "error" in df.columns:
                df["error"] = df["error"].fillna("").astype(str)

            # Coerce columns that are needed for grouping/filtering.
            for col in ["n", "rep", "alpha", "effect_strength"]:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors="coerce")

            if "reject" in df.columns:
                df["reject_num_preview"] = _coerce_reject_to_numeric(df["reject"])

            frames.append(df)

    if len(frames) == 0:
        return None

    combined = pd.concat(frames, ignore_index=True)
    return combined


all_results = load_all_available_results(OUT_DIR)
if all_results is None:
    print("No raw result files were found yet.")
else:
    print("Available methods:", sorted(all_results["method"].dropna().unique().tolist()))
    print("Rows loaded:", len(all_results))
    print(all_results["truth"].value_counts(dropna=False))
    print("Dtypes for key columns:")
    display(all_results[[c for c in ["method", "n", "effect_strength", "truth", "reject", "reject_num_preview", "error"] if c in all_results.columns]].dtypes)


In [ ]:
METHOD_LABELS = {
    "modified": "Revised mixed-data KCI",
    "onehot": "Standard KCI + one-hot",
}

if all_results is None or not {"modified", "onehot"}.issubset(set(all_results["method"].dropna().unique())):
    print("Run the notebook once with RUN_MODE='modified' and once with RUN_MODE='onehot', then rerun this cell.")
else:
    # Drop preview column if present; summarize_results creates its own reject_num.
    all_for_summary = all_results.drop(columns=["reject_num_preview"], errors="ignore")

    summary_all, query_summary_all, failures_all = summarize_results(all_for_summary)
    summary_all["scenario_label"] = summary_all["scenario"].map(SCENARIO_LABELS)
    summary_all["method_label"] = summary_all["method"].map(METHOD_LABELS)

    # Ensure key numeric columns are numeric before filtering/pivoting.
    for col in ["n", "effect_strength", "type1_error", "power"]:
        if col in summary_all.columns:
            summary_all[col] = pd.to_numeric(summary_all[col], errors="coerce")

    # Main paper-ready table: use the default empirical bandwidth rule.
    main_table = (
        summary_all[summary_all["est_width"] == "empirical"]
        .pivot_table(
            index=["scenario_label", "effect_strength", "effect_label", "n"],
            columns="method_label",
            values=["type1_error", "power"],
            aggfunc="first",
        )
        .sort_index()
    )
    display(main_table.round(3))
    main_table.to_csv(OUT_DIR / "ci_benchmark_main_table_empirical.csv")

    # Bandwidth sensitivity table: compare empirical vs median within each method.
    rows = []
    for (scenario, method, effect_strength, effect_label, n), g in summary_all.groupby(
        ["scenario_label", "method_label", "effect_strength", "effect_label", "n"]
    ):
        width_map = {row["est_width"]: row for _, row in g.iterrows()}
        if "empirical" in width_map and "median" in width_map:
            rows.append(
                {
                    "scenario": scenario,
                    "method": method,
                    "effect_strength": effect_strength,
                    "effect_label": effect_label,
                    "n": n,
                    "type1_error_empirical": width_map["empirical"].get("type1_error", np.nan),
                    "type1_error_median": width_map["median"].get("type1_error", np.nan),
                    "power_empirical": width_map["empirical"].get("power", np.nan),
                    "power_median": width_map["median"].get("power", np.nan),
                    "abs_diff_type1_error": abs(width_map["empirical"].get("type1_error", np.nan) - width_map["median"].get("type1_error", np.nan)),
                    "abs_diff_power": abs(width_map["empirical"].get("power", np.nan) - width_map["median"].get("power", np.nan)),
                }
            )
    sensitivity_table = pd.DataFrame(rows).sort_values(["scenario", "effect_strength", "n", "method"])
    display(sensitivity_table.round(3))
    sensitivity_table.to_csv(OUT_DIR / "ci_benchmark_bandwidth_sensitivity.csv", index=False)

    # Wide comparison table including both width rules.
    full_table = (
        summary_all
        .pivot_table(
            index=["scenario_label", "effect_strength", "effect_label", "n", "est_width"],
            columns="method_label",
            values=["type1_error", "power"],
            aggfunc="first",
        )
        .sort_index()
    )
    display(full_table.round(3))
    full_table.to_csv(OUT_DIR / "ci_benchmark_full_comparison.csv")

    # Compact table focused on n=2000 for the response letter.
    compact_n2000 = summary_all[
        (summary_all["n"] == 2000) &
        (summary_all["est_width"] == "empirical")
    ].pivot_table(
        index=["scenario_label", "effect_strength", "effect_label"],
        columns="method_label",
        values=["type1_error", "power"],
        aggfunc="first",
    ).sort_index()
    display(compact_n2000.round(3))
    compact_n2000.to_csv(OUT_DIR / "ci_benchmark_compact_n2000_empirical.csv")

    print("Saved:")
    print(" -", OUT_DIR / "ci_benchmark_main_table_empirical.csv")
    print(" -", OUT_DIR / "ci_benchmark_bandwidth_sensitivity.csv")
    print(" -", OUT_DIR / "ci_benchmark_full_comparison.csv")
    print(" -", OUT_DIR / "ci_benchmark_compact_n2000_empirical.csv")


In [ ]:
# Optional figure for the rebuttal / revision draft.
# This plots power and Type I error against n separately for each effect strength.
if all_results is None or not {"modified", "onehot"}.issubset(set(all_results["method"].dropna().unique())):
    print("Run both methods first, then rerun this cell.")
else:
    summary_all, _, _ = summarize_results(all_results)
    plot_df = summary_all[summary_all["est_width"] == "empirical"].copy()
    plot_df["scenario_label"] = plot_df["scenario"].map(SCENARIO_LABELS)
    plot_df["method_label"] = plot_df["method"].map(METHOD_LABELS)

    for effect_strength in sorted(plot_df["effect_strength"].unique()):
        sub_effect = plot_df[plot_df["effect_strength"] == effect_strength]
        scenario_order = [SCENARIO_LABELS[s] for s in SCENARIOS_TO_RUN]
        fig, axes = plt.subplots(2, len(scenario_order), figsize=(11, 6), sharex='col', sharey='row')
        if len(scenario_order) == 1:
            axes = np.array(axes).reshape(2, 1)

        for col, scenario_label in enumerate(scenario_order):
            sub = sub_effect[sub_effect["scenario_label"] == scenario_label].sort_values(["method_label", "n"])
            for method_label, g in sub.groupby("method_label"):
                axes[0, col].plot(g["n"], g["type1_error"], marker='o', label=method_label)
                axes[1, col].plot(g["n"], g["power"], marker='o', label=method_label)
            axes[0, col].axhline(ALPHA, linestyle='--', linewidth=1)
            axes[0, col].set_title(f"{scenario_label}\neffect={effect_strength}")
            axes[1, col].set_xlabel("Sample size")
            axes[0, col].set_ylabel("Type I error")
            axes[1, col].set_ylabel("Power")
            axes[0, col].set_ylim(0, 1)
            axes[1, col].set_ylim(0, 1)
            axes[0, col].legend(frameon=False)

        plt.tight_layout()
        fig_path = OUT_DIR / f"ci_benchmark_empirical_width_effect_{effect_strength}.png"
        plt.savefig(fig_path, dpi=200, bbox_inches='tight')
        plt.show()
        print("Saved figure to:", fig_path)
